# libs import

In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. SETUP PATHS & ENVIRONMENT

In [12]:
repo_dir = "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification"
src_path = os.path.join(repo_dir, "src")
if src_path not in sys.path:
  sys.path.append(src_path)

In [13]:
# Download dataset zip if missing
if not os.path.exists("imagesoasis.zip"):
  print("Downloading dataset from Kaggle...")
  !kaggle datasets download -d ninadaithal/imagesoasis
else:
  print("imagesoasis.zip already exists locally.")

# Unzip dataset inside content/
print("Unzipping dataset to local Colab SSD...")
os.makedirs("/content/dataset", exist_ok=True)
!unzip -qo imagesoasis.zip -d /content/dataset
print("Unzipped!")

'''VERIFY DATASET STRUCTURE & 4 CLASSES'''
# Check subdirectories to ensure all 4 classes (Mild, Moderate, Non, Very mild) exist
local_dataset_path = "/content/dataset/Data"
extracted_subdirs = []
for root, dirs, files in os.walk(local_dataset_path):
  for d in dirs:
    extracted_subdirs.append(d)

print("Detected folders/classes in dataset directory:", set(extracted_subdirs))

Dataset URL: https://www.kaggle.com/datasets/ninadaithal/imagesoasis
License(s): apache-2.0
100% 1.23G/1.23G [00:14<00:00, 88.7MB/s]

Unzipping dataset to local Colab SSD...
Unzipped!
Detected folders/classes in dataset directory: {'Mild Dementia', 'Very mild Dementia', 'Non Demented', 'Moderate Dementia'}


In [14]:
dataset_path = "/content/dataset/Data"
outputs_dir = os.path.join(repo_dir, "outputs")
os.makedirs(outputs_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# 2. LOAD DATA LOADERS & COMPUTE CLASS WEIGHTS

In [5]:
from dataset import make_stratified_splits

print("Initializing subject-level stratified data loaders...")
train_loader, val_loader, test_loader, split_info = make_stratified_splits(
  data_dir=dataset_path,
  batch_size=32,
  split_mode="subject"
)

# Count class frequencies in training set for alpha balancing
class_counts = [0, 0, 0, 0] # [Mild, Moderate, Non, Very mild]
for _, targets in train_loader:
  for t in targets:
    class_counts[t.item()] += 1

print(f"Training Class Counts: {class_counts}")

total_samples = sum(class_counts)
num_classes = len(class_counts)
# Inverse frequency weighting for Focal Loss alpha
class_weights = [total_samples / (num_classes * count) if count > 0 else 1.0 for count in class_counts]
alpha_tensor = torch.FloatTensor(class_weights).to(device)
print(f"Focal Loss Alpha Weights: {alpha_tensor}")

Initializing subject-level stratified data loaders...
Training Class Counts: [3599, 244, 46177, 9516]
Focal Loss Alpha Weights: tensor([ 4.1356, 61.0000,  0.3223,  1.5641], device='cuda:0')


# 3. DEFINE FOCAL LOSS CLASS

In [6]:
class FocalLoss(nn.Module):
  def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
    super(FocalLoss, self).__init__()
    self.alpha = alpha
    self.gamma = gamma
    self.reduction = reduction

  def forward(self, inputs, targets):
    ce_loss = F.cross_entropy(inputs, targets, reduction='none')
    pt = torch.exp(-ce_loss)
    focal_loss = ((1 - pt) ** self.gamma) * ce_loss

    if self.alpha is not None:
      alpha_t = self.alpha[targets]
      focal_loss = alpha_t * focal_loss

    if self.reduction == 'mean':
      return focal_loss.mean()
    elif self.reduction == 'sum':
      return focal_loss.sum()
    else:
      return focal_loss

# 4. INITIALIZE MODEL & OPTIMIZER

In [10]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 4)
model.to(device)

criterion = FocalLoss(alpha=alpha_tensor, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 5. TRAINING LOOP

In [ ]:
# Continue training
epochs = 12
best_val_acc = 0.0

print(f"\n--- STARTING TRAINING WITH FOCAL LOSS ({epochs} Epochs) ---")

for epoch in range(0,epochs):
  model.train()
  running_loss = 0.0
  correct_train = 0
  total_train = 0

  for batch_idx, (inputs, targets) in enumerate(train_loader):
    inputs, targets = inputs.to(device), targets.to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    _, predicted = outputs.max(1)
    total_train += targets.size(0)
    correct_train += predicted.eq(targets).sum().item()

    if batch_idx % 50 == 0:
      print(f"Epoch [{epoch+1}/{epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

  train_acc = correct_train / total_train
  train_loss = running_loss / len(train_loader)

  # Validation Phase
  model.eval()
  val_loss = 0.0
  correct_val = 0
  total_val = 0

  with torch.no_grad():
    for inputs, targets in val_loader:
      inputs, targets = inputs.to(device), targets.to(device)
      outputs = model(inputs)
      loss = criterion(outputs, targets)

      val_loss += loss.item()
      _, predicted = outputs.max(1)
      total_val += targets.size(0)
      correct_val += predicted.eq(targets).sum().item()

  val_acc = correct_val / total_val
  val_loss = val_loss / len(val_loader)

  print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}")

  # Save best checkpoint
  if val_acc > best_val_acc:
    best_val_acc = val_acc
    torch.save(model.state_dict(), os.path.join(outputs_dir, "best_model_focal.pt"))
    print(f"--> Saved New Best Focal Model with Val Acc: {val_acc:.4f}")

print("\nTraining complete! Best model saved to outputs/best_model_focal.pt")


--- STARTING TRAINING WITH FOCAL LOSS (12 Epochs) ---
Epoch [1/12] | Batch [0/1861] | Loss: 0.6996
Epoch [1/12] | Batch [50/1861] | Loss: 0.5276
Epoch [1/12] | Batch [100/1861] | Loss: 0.5520
Epoch [1/12] | Batch [150/1861] | Loss: 0.2782
Epoch [1/12] | Batch [200/1861] | Loss: 0.2031
Epoch [1/12] | Batch [250/1861] | Loss: 0.2327
Epoch [1/12] | Batch [300/1861] | Loss: 0.2104
Epoch [1/12] | Batch [350/1861] | Loss: 0.1940
Epoch [1/12] | Batch [400/1861] | Loss: 0.2867
Epoch [1/12] | Batch [450/1861] | Loss: 0.1822
Epoch [1/12] | Batch [500/1861] | Loss: 0.1315
Epoch [1/12] | Batch [550/1861] | Loss: 0.0991
Epoch [1/12] | Batch [600/1861] | Loss: 0.1791
Epoch [1/12] | Batch [650/1861] | Loss: 0.1285
Epoch [1/12] | Batch [700/1861] | Loss: 0.0928
Epoch [1/12] | Batch [750/1861] | Loss: 0.0805
Epoch [1/12] | Batch [800/1861] | Loss: 0.1573
Epoch [1/12] | Batch [850/1861] | Loss: 0.0797
Epoch [1/12] | Batch [900/1861] | Loss: 0.0638
Epoch [1/12] | Batch [950/1861] | Loss: 0.1566
Epoch [1

# 6. Load newly trained Focal Model weights

In [10]:
repo_dir = "/content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification"
src_path = os.path.join(repo_dir, "src")
if src_path not in sys.path:
  sys.path.append(src_path)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 4)


focal_model_path = os.path.join(repo_dir, "outputs/best_model_focal.pt")
if os.path.exists(focal_model_path):
  print(f"Loading weights from: {focal_model_path}")
  state_dict = torch.load(focal_model_path, map_location=device)
  model.load_state_dict(state_dict)
else:
  print("Error: best_model_focal.pt not found! Check outputs folder.")

Loading weights from: /content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification/outputs/best_model_focal.pt


In [5]:
model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

# 7. Run inference on test set

In [15]:
from dataset import make_stratified_splits

train_loader, val_loader, test_loader, split_info = make_stratified_splits(
  data_dir=dataset_path,
  batch_size=32,
  split_mode="subject"
)

In [16]:
all_preds = []
all_targets = []
class_names = ['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']

print("Running inference with Focal model...")
with torch.no_grad():
  for inputs, targets in test_loader:
    inputs, targets = inputs.to(device), targets.to(device)
    outputs = model(inputs)
    _, preds = torch.max(outputs, 1)
    all_preds.extend(preds.cpu().numpy())
    all_targets.extend(targets.cpu().numpy())

Running inference with Focal model...


# 8. Reports

In [17]:
print("\n=== FOCAL MODEL TEST CLASSIFICATION REPORT ===")
report = classification_report(all_targets, all_preds, target_names=class_names, zero_division=0)
print(report)

# Save report to outputs
outputs_dir = os.path.join(repo_dir, "outputs")
with open(os.path.join(outputs_dir, "classification_report_focal_test.txt"), "w") as f:
  f.write(report)


=== FOCAL MODEL TEST CLASSIFICATION REPORT ===
                    precision    recall  f1-score   support

     Mild Dementia       0.14      0.03      0.04       732
 Moderate Dementia       0.67      0.01      0.02       244
      Non Demented       0.77      0.96      0.86     10126
Very mild Dementia       0.34      0.07      0.12      2135

          accuracy                           0.75     13237
         macro avg       0.48      0.27      0.26     13237
      weighted avg       0.67      0.75      0.68     13237



## Confusion matrix

In [18]:
plt.figure(figsize=(8, 6))
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Focal Model Test Confusion Matrix')
cm_path = os.path.join(outputs_dir, "confusion_matrix_focal_test.png")
plt.savefig(cm_path)
plt.close()
print(f"\nFocal confusion matrix saved successfully to: {cm_path}")


Focal confusion matrix saved successfully to: /content/drive/MyDrive/Colab Notebooks/Oasis-based-dementia-classification/outputs/confusion_matrix_focal_test.png
